# Airline Satisfaction — CatBoost baseline

This notebook trains a binary classifier for `satisfaction`, evaluates it with **ROC AUC** on a stratified holdout, then fits on all labeled rows and writes `submission_catboost.csv`. It assumes this notebook and the two CSV files are in the same folder.


In [2]:
# Run once if CatBoost is missing from this notebook environment.
%pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [2]:
# Files are next to this notebook. The training CSV was named "train (1).csv".
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = "satisfaction"
ID_COL = "id"

print("train:", train.shape, "test:", test.shape)
display(train.head())
print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False))
print("\nMissing values (training):")
print(train.isna().sum().sort_values(ascending=False).head(10))


train: (699635, 23) test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False



Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Missing values (training):
Arrival Delay in Minutes      292
id                              0
Leg room service                0
Class                           0
Type of Travel                  0
Customer Type                   0
Gender                          0
Departure Delay in Minutes      0
Cleanliness                     0
Checkin service                 0
dtype: int64


In [3]:
# Separate identifiers and target; do not train on the ID by default.
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

# CatBoost accepts categorical columns directly; represent missing categories explicitly.
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns use consistent numeric values and preserve missing values for CatBoost.
for col in X.columns.difference(cat_features):
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical features:", cat_features)
print("Positive class rate:", y.mean())


Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Positive class rate: 0.44357272006117476


In [4]:
# Compare the current features against one engineered travel-context feature.
# Both models use exactly the same train/validation rows and CatBoost settings.
from sklearn.model_selection import train_test_split

row_indices = np.arange(len(y))
train_idx, valid_idx = train_test_split(
    row_indices, test_size=0.2, random_state=42, stratify=y
)
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

# Baseline feature tables: no engineered column.
X_base = X.copy()
X_test_base = X_test.copy()

# Engineered feature: combine three related categorical passenger context fields.
X_context = X.copy()
X_test_context = X_test.copy()
context_cols = ["Type of Travel", "Customer Type", "Class"]
X_context["Travel_Context"] = X_context[context_cols].astype(str).agg(" | ".join, axis=1)
X_test_context["Travel_Context"] = X_test_context[context_cols].astype(str).agg(" | ".join, axis=1)

feature_sets = {
    "baseline": (X_base, X_test_base, cat_features.copy()),
    "with_travel_context": (X_context, X_test_context, cat_features + ["Travel_Context"]),
}
models_by_features = {}
scores_by_features = {}

for feature_name, (features, _, categorical_cols) in feature_sets.items():
    print(f"\nTraining CatBoost: {feature_name}")
    candidate = CatBoostClassifier(
        iterations=4000,
        learning_rate=0.03,
        depth=7,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=300,
        early_stopping_rounds=150,
        allow_writing_files=False,
        thread_count=-1,
    )
    candidate.fit(
        features.iloc[train_idx], y_train,
        cat_features=categorical_cols,
        eval_set=(features.iloc[valid_idx], y_valid),
        use_best_model=True,
    )
    candidate_proba = candidate.predict_proba(features.iloc[valid_idx])[:, 1]
    candidate_auc = roc_auc_score(y_valid, candidate_proba)
    models_by_features[feature_name] = candidate
    scores_by_features[feature_name] = candidate_auc
    print(f"{feature_name}: validation ROC AUC={candidate_auc:.6f}; best iteration={candidate.best_iteration_}")

best_feature_name = max(scores_by_features, key=scores_by_features.get)
model = models_by_features[best_feature_name]
valid_proba = model.predict_proba(feature_sets[best_feature_name][0].iloc[valid_idx])[:, 1]
X_final, X_test_final, final_cat_features = feature_sets[best_feature_name]
best_depth = 7
print(f"\nSelected feature set: {best_feature_name} (AUC={scores_by_features[best_feature_name]:.6f})")



Training CatBoost: baseline
0:	test: 0.9374761	best: 0.9374761 (0)	total: 400ms	remaining: 26m 39s
300:	test: 0.9557906	best: 0.9557906 (300)	total: 1m 17s	remaining: 15m 55s
600:	test: 0.9568228	best: 0.9568228 (600)	total: 2m 44s	remaining: 15m 32s
900:	test: 0.9571934	best: 0.9571934 (900)	total: 4m 14s	remaining: 14m 34s
1200:	test: 0.9573872	best: 0.9573879 (1199)	total: 5m 42s	remaining: 13m 17s
1500:	test: 0.9575116	best: 0.9575131 (1499)	total: 7m 11s	remaining: 11m 58s
1800:	test: 0.9575765	best: 0.9575772 (1785)	total: 8m 42s	remaining: 10m 37s
2100:	test: 0.9576408	best: 0.9576408 (2100)	total: 10m 11s	remaining: 9m 12s
2400:	test: 0.9576786	best: 0.9576827 (2339)	total: 11m 10s	remaining: 7m 26s
2700:	test: 0.9577260	best: 0.9577282 (2685)	total: 12m 17s	remaining: 5m 54s
3000:	test: 0.9577445	best: 0.9577463 (2981)	total: 13m 14s	remaining: 4m 24s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9577677706
bestIteration = 3139

Shrink model to first 31

## Optional threshold-based check

 The competition is scored by ROC AUC, so probability ranking is the key result. The next cell is only an optional classification report at threshold 0.5.


In [5]:
valid_pred = (valid_proba >= 0.5).astype(int)
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, valid_pred))


              precision    recall  f1-score   support

           0     0.9239    0.9436    0.9336     77859
           1     0.9273    0.9025    0.9147     62068

    accuracy                         0.9254    139927
   macro avg     0.9256    0.9231    0.9242    139927
weighted avg     0.9254    0.9254    0.9253    139927

Confusion matrix:
 [[73467  4392]
 [ 6050 56018]]


In [6]:
# Refit on all labeled rows using the feature set selected by validation.
best_iterations = model.best_iteration_ + 1 if model.best_iteration_ is not None else None
if best_iterations is None or best_iterations < 1:
    best_iterations = 1500

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.03,
    depth=best_depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)
final_model.fit(X_final, y, cat_features=final_cat_features)
test_proba = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_catboost.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())


0:	total: 252ms	remaining: 16m 49s
200:	total: 1m 11s	remaining: 22m 27s
400:	total: 2m 24s	remaining: 21m 35s
600:	total: 3m 35s	remaining: 20m 17s
800:	total: 4m 41s	remaining: 18m 45s
1000:	total: 5m 48s	remaining: 17m 22s
1200:	total: 6m 55s	remaining: 16m 9s
1400:	total: 8m 5s	remaining: 15m
1600:	total: 9m 15s	remaining: 13m 52s
1800:	total: 10m 21s	remaining: 12m 38s
2000:	total: 11m 33s	remaining: 11m 31s
2200:	total: 12m 45s	remaining: 10m 25s
2400:	total: 14m 3s	remaining: 9m 21s
2600:	total: 15m 20s	remaining: 8m 14s
2800:	total: 16m 34s	remaining: 7m 5s
3000:	total: 17m 45s	remaining: 5m 54s
3200:	total: 18m 56s	remaining: 4m 43s
3400:	total: 20m 8s	remaining: 3m 32s
3600:	total: 21m 28s	remaining: 2m 22s
3800:	total: 22m 47s	remaining: 1m 11s
3998:	total: 24m 2s	remaining: 0us
Saved: C:\Users\Asus\OneDrive\Desktop\Predicting_Airline_Satisfaction\submission_catboost.csv
       id  satisfaction
0  699635      0.222305
1  699636      0.881760
2  699637      0.909788
3  699638